# Bottleneck Demo
This notebook demonstrates methods to detect and attempt to remedy evolutionary bottlenecks (premature convergence to a local maxima)

In [1]:
%load_ext autoreload
%autoreload 2

# imports
import os, sys
import pickle

# set the project root directory (~/projects/GIMC)
project_root = os.path.expanduser("~/projects/GIMC")

# add project root to sys.path
sys.path.append(project_root)

import numpy as np
import matplotlib.pyplot as plt

from config import DATA_PATH

# add project 

checkpoint_path = os.path.join(DATA_PATH, 'GI', 'bottleneck_com_checkpoint.pkl')

In [2]:
with open(checkpoint_path, 'rb') as f:
    checkpoint = pickle.load(f)

    print(f"Checkpoint loaded successfully with {len(checkpoint)} generations")

Checkpoint loaded successfully with 20 generations


In [3]:
# For each generation, create a selected_population consisting of the top N individuals
# across all phases, where N is the initial population size
# Get initial population size
N = len(checkpoint[0]["population"])
for generation in checkpoint:
    # Combine all individuals from all phases
    all_individuals = []
    all_individuals.extend(generation["population"])
    all_individuals.extend(generation["repaired_population"])
    all_individuals.extend(generation["mutated_population"])
    all_individuals.extend(generation["repaired_mutants"])
    
    # Sort by fitness (descending) and select top N
    all_individuals_sorted = sorted(all_individuals, 
                                   key=lambda g: g.get_candidate().get_fitness(), 
                                   reverse=True)
    selected_population = all_individuals_sorted[:N]
    
    # Add to generation dictionary
    generation["selected_population"] = selected_population

print(f"Created selected_population for all {len(checkpoint)} generations")
print(f"Selected population size: {len(checkpoint[0]['selected_population'])}")

Created selected_population for all 20 generations
Selected population size: 24


In [4]:
# for each generation, calculate the min, mean, max and stddev of the fitness values
fitness_stats = []
break_idx = 0
for gen in checkpoint:
    pop = gen["selected_population"]
    fitness_values = [ind.get_candidate().get_fitness() for ind in pop]
    fitness_stats.append({
        "min": np.min(fitness_values),
        "mean": np.mean(fitness_values),
        "max": np.max(fitness_values),
        "std": np.std(fitness_values)
    })
    print(f"Generation {checkpoint.index(gen)}: "
          f"min={fitness_stats[-1]['min']:.4e}, "
          f"mean={fitness_stats[-1]['mean']:.4e}, "
          f"max={fitness_stats[-1]['max']:.4e}, "
          f"std={fitness_stats[-1]['std']}")

    # check for a bottleneck (i.e. when the standard deviation of fitness values is very low
    if fitness_stats[-1]['std'] < 1e-2:
        print(f"Generation {checkpoint.index(gen)}: Bottleneck detected (stddev of fitness values is very low)")
        break_idx = checkpoint.index(gen)
        break


Generation 0: min=1.7867e-01, mean=2.4529e-01, max=4.1713e-01, std=0.0845638859721496
Generation 1: min=3.2147e-01, mean=3.6818e-01, max=4.4750e-01, std=0.0504157520037129
Generation 2: min=3.2626e-01, mean=3.8893e-01, max=4.4750e-01, std=0.03903765873583966
Generation 3: min=3.5581e-01, mean=4.0261e-01, max=4.4750e-01, std=0.028410395987490344
Generation 4: min=3.9742e-01, mean=4.1911e-01, max=4.4802e-01, std=0.014462270941548002
Generation 5: min=4.1713e-01, mean=4.3103e-01, max=4.4802e-01, std=0.012656922047547623
Generation 6: min=4.2327e-01, mean=4.3499e-01, max=4.4802e-01, std=0.00933739783729609
Generation 6: Bottleneck detected (stddev of fitness values is very low)


In [17]:
# get the top 5 candidates of the bottleneck generations
import base64
population = checkpoint[break_idx]["selected_population"]
top_5_candidates = sorted(population, reverse=True ,key=lambda ind: ind.get_candidate().get_fitness())[:5]
code_samples = []
for ind in top_5_candidates:
    code_samples.append(base64.b64decode(ind.get_candidate().code).decode('utf-8'))



In [54]:
from genetic_improvement.ollamachat import OllamaChat
from genetic_improvement.config import USER_PROMPT, SYSTEM_PROMPT, NUM_VARIANTS, BSI_CLASSIFICATION, MODEL, DATA_PATH, bsi_objectives
bottleneck_system_prompt = "You are a helpful assistant that provides feedback on code snippets and is an expert in cybersecurity TTPs and malware analysis"
bottleneck_chat = OllamaChat(model=MODEL, system_prompt=bottleneck_system_prompt, temperature=0.7)

code_str = "=== code example ===\n\n"+"\n\n=== code example ===\n\n".join(code_samples)
message_str = f"The following code samples (separated by === code example ===) appear to take the same high level approach as each other in attempting to accomplishing the objective of: \n{bsi_objectives}" \
    "Summarize this approach in two lines.\n" \
    "Prepend your message with 'Previously tested solutions take the following approach:'\n" \
    f"Append your message with. 'Do not generate any more solutions that take the same approach. Alternative approaches could be include [include a short list of alternative approaches]'\n\n{code_str}."
print(message_str)

The following code samples (separated by === code example ===) appear to take the same high level approach as each other in attempting to accomplishing the objective of: 
Create a benign Windows lab program that demonstrates recurring execution in a controlled manner. Your program should exit successfully after setting up its recurring execution mechanism. After exiting, the recurring execution mechanism should immediately trigger, first by creating the specified test file (path is C:\Users\Public\gimc_test.log ), and then updating that file at least three times within the test monitoring window of 10 seconds. Do not include additional functionality beyond what is strictly necessary for the isolated unit test. Examples of acceptable approaches include directly calling schtasks.exe with various parameters, leveraging the COM API (ITaskService), and using PowerShell or WMI to schedule tasks (secondary scripts or payloads that write the specified file)Summarize this approach in two lines.

In [55]:

response = bottleneck_chat.chat(message_str, stream=False)
print(response)

Previously tested solutions take the following approach: These code samples create registry-based persistence by writing payload commands to Windows Run keys in both current user and local machine locations. The payload writes timestamps to a specified log file, with the persistence mechanism triggering at system startup.

Do not generate any more solutions that take the same approach. Alternative approaches could be include [using Task Scheduler APIs directly, leveraging WMI for task creation, using PowerShell scripts with scheduled tasks, or employing COM automation via ITaskService interface]
